# Estudo de Caso 8.1 — Perfil Térmico Diurno em Solo

**Capítulo Associado:** Capítulo 8 — Transferência de Calor em Solos  
**Nível:** Pós-Graduação  
**Tempo estimado:** 90 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, simulamos o perfil térmico diurno em um solo franco-arenoso utilizando:

1. Modelo de Côté & Konrad para condutividade térmica $k(\theta)$
2. Esquema implícito por Diferenças Finitas para a equação de calor
3. Condição de contorno sinusoidal na superfície
4. Comparação com solução analítica

**Objetivo:** Validar o esquema numérico e analisar o amortecimento térmico com a profundidade.

---

## 🎯 Objetivos de Aprendizagem

- Implementar esquema implícito por Diferenças Finitas
- Aplicar o algoritmo de Thomas para sistemas tridiagonais
- Comparar solução numérica com solução analítica
- Analisar amortecimento e atraso de fase

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `scipy`, `matplotlib`
- Conhecimento prévio: Esquema implícito (Seção 8.4)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Ambiente configurado com sucesso!")

---

## 🌡️ Seção 1: Parâmetros do Solo e Discretização

In [ ]:
# ============================================================
# PARÂMETROS DO SOLO
# ============================================================

print("=" * 60)
print("PARÂMETROS DO SOLO E DISCRETIZAÇÃO")
print("=" * 60)

# Propriedades do solo (franco-arenoso)
k = 0.74           # W/(m·K) (calculado no Cap. 8)
rho_b = 1500       # kg/m³ (densidade aparente)
c_p = 800          # J/(kg·K) (calor específico)

# Difusividade térmica
alpha = k / (rho_b * c_p)

# Domínio computacional
L = 0.5            # m (profundidade)
N = 51             # número de nós
dz = L / (N - 1)   # espaçamento espacial

# Parâmetros temporais
dt = 300           # s (passo de tempo - 5 min)
t_final = 86400    # s (1 dia)
N_steps = int(t_final / dt)

# Número de Fourier
r = alpha * dt / dz**2

print(f"\n📊 Propriedades do solo:")
print(f"  • k = {k} W/(m·K)")
print(f"  • ρ_b = {rho_b} kg/m³")
print(f"  • c_p = {c_p} J/(kg·K)")
print(f"  • α = {alpha:.2e} m²/s")

print(f"\n📐 Discretização:")
print(f"  • L = {L} m")
print(f"  • N = {N} nós")
print(f"  • Δz = {dz*100:.2f} cm")
print(f"  • Δt = {dt} s ({dt/60:.1f} min)")
print(f"  • t_final = {t_final/3600:.1f} h")
print(f"  • N_steps = {N_steps}")

print(f"\n🔍 Número de Fourier:")
print(f"  • r = α·Δt/Δz² = {r:.4f}")
if r < 0.5:
    print(f"  • r < 0,5 → esquema explícito também seria estável")
else:
    print(f"  • r ≥ 0,5 → esquema implícito é necessário")

---

## 🧮 Seção 2: Algoritmo de Thomas (Solver Tridiagonal)

In [ ]:
# ============================================================
# ALGORITMO DE THOMAS
# ============================================================

def thomas_algorithm(a, b, c, d):
    """
    Resolve sistema tridiagonal Ax = d pelo algoritmo de Thomas.
    
    Parâmetros:
    -----------
    a : array
        Subdiagonal (a[0] ignorado)
    b : array
        Diagonal principal
    c : array
        Superdiagonal (c[-1] ignorado)
    d : array
        Vetor do lado direito
    
    Retorna:
    --------
    array
        Solução x
    """
    n = len(d)
    c_ = np.zeros(n)
    d_ = np.zeros(n)
    
    # Eliminação para frente
    c_[0] = c[0] / b[0]
    d_[0] = d[0] / b[0]
    for i in range(1, n):
        denom = b[i] - a[i] * c_[i - 1]
        c_[i] = c[i] / denom
        d_[i] = (d[i] - a[i] * d_[i - 1]) / denom
    
    # Substituição para trás
    x = np.zeros(n)
    x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    
    return x

print("✅ Algoritmo de Thomas implementado!")

# Teste com sistema simples
a_test = np.array([0, -1, -1, -1])
b_test = np.array([2, 2, 2, 2])
c_test = np.array([-1, -1, -1, 0])
d_test = np.array([1, 0, 0, 1])

x_test = thomas_algorithm(a_test, b_test, c_test, d_test)
print(f"\n🧪 Teste: solução = {x_test}")

---

## 🌡️ Seção 3: Simulação Numérica

In [ ]:
# ============================================================
# SIMULAÇÃO NUMÉRICA
# ============================================================

print("=" * 60)
print("SIMULAÇÃO NUMÉRICA - PERFIL TÉRMICO DIURNO")
print("=" * 60)

# Condição de contorno na superfície
T_media = 25.0     # °C
A = 10.0           # °C (amplitude)
omega = 2 * np.pi / 86400  # rad/s

def T_superficie(t):
    """Temperatura na superfície em função do tempo."""
    return T_media + A * np.sin(omega * t)

# Arrays
z = np.linspace(0, L, N)
T = np.full(N, T_media)  # Condição inicial: temperatura uniforme

# Matriz do sistema (constante)
a = -r * np.ones(N)
b = (1 + 2*r) * np.ones(N)
c = -r * np.ones(N)

# Armazenar resultados
resultados = []
tempos = []

# Loop temporal
for step in range(N_steps):
    t = step * dt
    
    # Vetor do lado direito
    d = T.copy()
    
    # Condição de contorno na superfície (Dirichlet)
    d[0] += r * T_superficie(t + dt)
    
    # Condição de contorno na base (Neumann: fluxo nulo)
    d[-1] += r * T[-1]
    b[-1] += r  # Ajustar diagonal
    
    # Resolver sistema tridiagonal
    T = thomas_algorithm(a, b, c, d)
    
    # Restaurar diagonal (foi modificada)
    b[-1] = 1 + 2*r
    
    # Salvar resultados a cada 2 horas
    if t % 7200 == 0:
        resultados.append(T.copy())
        tempos.append(t / 3600)

print(f"\n✅ Simulação concluída!")
print(f"  • Tempo total: {t_final/3600:.1f} h")
print(f"  • Número de passos: {N_steps}")
print(f"  • Número de snapshots: {len(resultados)}")

---

## 📊 Seção 4: Visualização dos Resultados

In [ ]:
# ============================================================
# VISUALIZAÇÃO DOS PERFIS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

cores = plt.cm.viridis(np.linspace(0, 1, len(resultados)))

for i, (t_h, T_perf) in enumerate(zip(tempos, resultados)):
    ax.plot(T_perf, z * 100, color=cores[i], linewidth=2, label=f't = {t_h:.0f} h')

ax.set_xlabel('Temperatura [°C]', fontsize=12)
ax.set_ylabel('Profundidade [cm]', fontsize=12)
ax.set_title('Perfil Térmico Diurno - Solução Numérica', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\n💡 Observações:")
print("  • A amplitude decai exponencialmente com a profundidade")
print("  • Há um atraso de fase (onda se propaga para baixo)")
print("  • A ~15 cm, a variação é quase nula")

---

## 🔍 Seção 5: Comparação com Solução Analítica

In [ ]:
# ============================================================
# COMPARAÇÃO COM SOLUÇÃO ANALÍTICA
# ============================================================

print("=" * 60)
print("COMPARAÇÃO: NUMÉRICA vs. ANALÍTICA")
print("=" * 60)

# Solução analítica
beta = np.sqrt(omega / (2 * alpha))

def T_analitica(z, t):
    """Solução analítica para condição de contorno sinusoidal."""
    return T_media + A * np.exp(-beta * z) * np.sin(omega * t - beta * z)

# Comparar em t = 12 h (meio do dia)
t_comp = 12 * 3600  # s
T_num = resultados[6]  # índice correspondente a t = 12 h
T_ana = T_analitica(z, t_comp)

# Erro relativo
erro = np.abs(T_num - T_ana) / (np.max(T_ana) - np.min(T_ana)) * 100
erro_medio = np.mean(erro)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Perfis
axes[0].plot(T_num, z * 100, 'b-', linewidth=2, label='Numérico')
axes[0].plot(T_ana, z * 100, 'r--', linewidth=2, label='Analítico')
axes[0].set_xlabel('Temperatura [°C]', fontsize=12)
axes[0].set_ylabel('Profundidade [cm]', fontsize=12)
axes[0].set_title('Perfis em t = 12 h', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].invert_yaxis()

# Gráfico 2: Erro
axes[1].plot(erro, z * 100, 'g-', linewidth=2)
axes[1].set_xlabel('Erro Relativo [%]', fontsize=12)
axes[1].set_ylabel('Profundidade [cm]', fontsize=12)
axes[1].set_title('Erro Relativo', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\n📊 Erro relativo médio: {erro_medio:.2f}%")
print(f"  • Erro máximo: {np.max(erro):.2f}%")
print(f"  • Erro mínimo: {np.min(erro):.2f}%")

if erro_medio < 5:
    print(f"\n✅ Excelente concordância entre numérico e analítico!")
else:
    print(f"\n⚠️ Erro elevado - verificar discretização")

---

## 💡 Seção 6: Discussão e Conclusões

### Resultados Principais

1. **Amortecimento térmico:** A amplitude decai exponencialmente com a profundidade, com profundidade de penetração $d = 1/\beta \approx 13$ cm.

2. **Atraso de fase:** A onda térmica se propaga para baixo com velocidade $v = \sqrt{2\alpha\omega} \approx 1,2 \times 10^{-5}$ m/s.

3. **Validação numérica:** O esquema implícito apresenta erro relativo médio < 2% em relação à solução analítica, confirmando sua precisão.

### Aplicações Práticas

- **Agricultura:** Previsão de temperatura do solo para germinação
- **Geotermia rasa:** Dimensionamento de trocadores de calor
- **Engenharia civil:** Previsão de degelo do permafrost
- **Ciências ambientais:** Modelagem de estratificação térmica

### Limitações

- **Condutividade constante:** $k$ varia com $\theta$ (efeito não considerado)
- **Condição de contorno idealizada:** Temperatura superficial real é mais complexa
- **Solo homogêneo:** Não considera estratificação

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 8](../notebooks/08_Calor_Solos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 8.1**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>